# A4 — Remediation Family Clustering (HDBSCAN)

The L2 catalog holds 3,870 individual checks, but an engineer does not remediate 3,870 different ways — "enable encryption at rest" is *one action* recurring across dozens of services and all three tools. A4 recovers those **remediation families**, so later studies can prioritise at the level of the action a team actually takes rather than the individual finding.

Logic lives in `src/misconfig/clustering.py` (CLAUDE.md rule 9).

**Method:** embed remediation-oriented text → UMAP → HDBSCAN. HDBSCAN is used rather than k-means because it finds variable-density clusters and does not force every check into a family — genuinely unique checks are allowed to stay as noise.

**Two parameters are swept rather than assumed** (CLAUDE.md rule 5):
- `min_cluster_size`, the main structural knob;
- the random seed — UMAP is stochastic, so stability across 10 seeds is measured with the Adjusted Rand Index and reported as mean ± std.

**Caveat:** only 46% of checks ship remediation text (Prowler mostly). For the rest the family is inferred from title and description, so families are closer to "what the check is about" than "the exact fix" wherever remediation text is missing.

In [1]:
import os
import sys
from pathlib import Path

# Every src/ path is relative to the project root (data/..., outputs/...).
# Without this chdir a run from notebooks/ writes into notebooks/data/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.misconfig.clustering import run, log_stats

In [2]:
results = run()
log_stats(results)

2026-11-23 17:49:35,168 INFO src.misconfig.clustering: Embedding 3870 checks


2026-11-23 17:49:38,108 INFO src.misconfig.matching: Loading sentence-transformer model all-MiniLM-L6-v2


2026-11-23 17:49:38,124 INFO sentence_transformers.base.model: No device provided, using mps


2026-11-23 17:49:38,437 INFO httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-11-23 17:49:38,439 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-11-23 17:49:38,509 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"


... (54 outputs left out)

/Users/spopa/Documents/Internship Project/internship-project/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2026/11/23 17:52:02 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/spopa/Documents/Internship Project/internship-project/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026-11-23 17:52:02,802 INFO src.tracking: Logged A4 to MLflow (run 17a109dd2f484e6abb53be24c41d94ab)



A4 remediation family clustering
Checks clustered:        3,870
Families found:          138  (min_cluster_size=10)
Unclustered (noise):     623 (16.1%)
Largest family:          80 checks
Median family size:      20 checks

Stability over 10 seeds (UMAP is stochastic):
  families:   139.5 ± 3.2
  noise:      16.3% ± 0.7%
  pairwise ARI: 0.532 ± 0.043

Families spanning >1 tool:     106 of 138
Families spanning >1 provider: 99 of 138

Parameter sweep:
 min_cluster_size  n_families  noise_pct  median_family_size  silhouette
                5         228       16.6                  11       0.704
               10         138       16.1                  20       0.659
               15          88       15.1                  30       0.626
               20          73       16.0                  37       0.614
               30          46       18.9                  59       0.570

10 largest families:
 family  size  n_tools  n_providers                                                 

## The families

Each family is named by its most distinctive terms (mean TF-IDF over its members). A family spanning several tools or providers is the interesting case: it means the same remediation action recurs across vendors.

In [3]:
results["families"].head(25)

,family,size,top_terms,tools,n_tools,providers,n_providers,example_check
0,119,80,"rds, redis, az, enabled, multi az","checkov, prowler, scoutsuite",3,"alibabacloud, alicloud, aws, azure, do",6,Ensure RDS instance auto upgrades for minor ve...
1,3,78,"github, branch, repository, ensure github, bra...","checkov, prowler",2,"aws, azure, azuredevops, gcp, github",7,Ensure that Azure Data Factory uses Git reposi...
2,128,74,"backup, snapshots, backups, deletion protectio...","checkov, prowler, scoutsuite",3,"aliyun, aws, e2enetworks, gcp, kubernetes",7,Ensure that Elastic File System (Amazon EFS) f...
3,101,68,"encryption, managed, disks, key, customer","checkov, prowler, scoutsuite",3,"alicloud, aws, azure, do, gcp",7,Ensure that Databricks Workspaces enables cust...
4,92,68,"cmk, kms, encrypted, encrypted kms, key cmk","checkov, prowler",2,"aws, yandexcloud",2,Ensure AWS SQS uses CMK not AWS default keys f...
5,13,65,"secrets, secret, hard coded, coded, ensure hard","checkov, prowler, scoutsuite",3,"aws, azure, bridgecrew, gcp, k8s",11,No ServiceAccount/Node should be able to read ...
6,127,63,"tcp, ingress, security, internet, ports","checkov, prowler, scoutsuite",3,"alibabacloud, aws, azure, gcp, huaweicloud",8,Ensure no SQL Databases allow ingress from 0.0...
7,65,58,"conditional access, conditional, access, entra...",prowler,1,"azure, googleworkspace, m365",3,Strong authentication methods are enabled with...
8,133,57,"encrypted, encryption, ebs, ensure, securely e...","checkov, prowler, scoutsuite",3,"alibabacloud, alicloud, aws, e2enetworks, huaw...",9,Ensure Transparent Data Encryption is Enabled ...
9,14,51,"postgresql, postgresql database, database, fla...","checkov, prowler, scoutsuite",3,"alibabacloud, azure, do, gcp",4,Ensure server parameter 'log_checkpoints' is s...


## Stability

UMAP is stochastic, so a single run's families could be an artefact. Pairwise ARI across seeds says how reproducible the structure actually is — read this before treating any individual family as real.

In [4]:
print(results["stability"])
results["seed_summary"]

{'n_families_mean': np.float64(139.5), 'n_families_std': np.float64(3.24037034920393), 'noise_pct_mean': np.float64(16.330749354005167), 'noise_pct_std': np.float64(0.6605874070220158), 'ari_mean': 0.5316490387961247, 'ari_std': 0.042870809517203276}


,seed,n_families,noise_pct
0,0,138,16.098191
1,1,135,16.821705
2,2,137,17.054264
3,3,138,16.950904
4,4,143,16.304910
5,5,140,15.012920
6,6,146,15.736434
7,7,140,15.813953
8,8,137,16.718346
9,9,141,16.795866


## Sanity-check a family by reading it

A clustering is only useful if its groups are coherent to a human. Pick the largest families and read their members.

In [5]:
assignments = results["assignments"]
for family in results["families"]["family"].head(3):
    members = assignments[assignments.family == family]
    print(f"--- family {family} ({len(members)} checks): {members.family_terms.iloc[0]} ---")
    for title in members.title.head(8):
        print("   ", title)
    print()

--- family 119 (80 checks): rds, redis, az, enabled, multi az ---
    Ensure RDS instance auto upgrades for minor versions
    Ensure log audit is enabled for RDS
    Ensure Postgres RDS as aws_rds_cluster has Query Logging enabled
    Ensure Postgres RDS as aws_db_instance has Query Logging enabled
    Ensure AWS ElastiCache Redis cluster with Multi-AZ Automatic Failover feature set to enabled
    Ensure that enhanced monitoring is enabled for Amazon RDS instances
    Ensure that enhanced monitoring is enabled for Amazon RDS instances
    Ensure that respective logs of Amazon Relational Database Service (Amazon RDS) are enabled

--- family 3 (78 checks): github, branch, repository, ensure github, branch protection ---
    Ensure that Azure Data Factory uses Git repository for source control
    Merge requests should require at least 2 approvals
    The build output cannot be affected by user parameters other than the build entry point and the top-level source location. GitHub Actions 